# Module 05 — Neural Network Foundations
## 05-04: Loss Functions

**Objective:** Implement MSE, MAE, Huber, cross-entropy, and focal loss from scratch; understand their gradients, robustness properties, and which training scenarios each suits best.

**Prerequisites:** 05-01 (Perceptron), 05-02 (MLP), 05-03 (Activation Functions)

## Part 0 — Setup & Prerequisites

This notebook covers the **loss function landscape** — the objective functions that measure prediction error and drive gradient descent. A loss function's geometry determines what kinds of errors the model is penalised for and how strongly, which directly shapes what the trained model learns to optimise.

We implement five loss functions from scratch in NumPy, study their gradients and robustness properties, wrap them as PyTorch `nn.Module` classes, and run training experiments on MNIST (classification) and synthetic regression data with outliers.

**Prerequisites:** 05-01 (Perceptron), 05-02 (MLP), 05-03 (Activation Functions)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

import random
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility & Device ─────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
BATCH_SIZE      = 256
NUM_EPOCHS      = 8          # For classification comparison runs
NUM_EPOCHS_REG  = 200        # For synthetic regression experiments
LEARNING_RATE   = 1e-3
DATA_DIR        = "../../data"
INPUT_DIM       = 784        # MNIST: 28×28 flattened
NUM_CLASSES     = 10
HIDDEN_DIM      = 256
HUBER_DELTA     = 1.0        # Huber transition point
FOCAL_GAMMA     = 2.0        # Focal loss focusing parameter

DIGIT_NAMES = ["0","1","2","3","4","5","6","7","8","9"]

print(f"MNIST: {INPUT_DIM}-D input → {NUM_CLASSES} classes  hidden={HIDDEN_DIM}")
print(f"Huber delta={HUBER_DELTA}  Focal gamma={FOCAL_GAMMA}")

### Data Loading & EDA

We use **MNIST** (handwritten digits 0–9) for classification experiments and synthetically generated 1-D data with outliers for regression comparisons.

In [ ]:
# ── MNIST (official train/test splits) ────────────────────────────────────────
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])
full_train_set = datasets.MNIST(
    root=DATA_DIR, train=True, download=True, transform=_transform
)
test_set = datasets.MNIST(
    root=DATA_DIR, train=False, download=True, transform=_transform
)
generator  = torch.Generator().manual_seed(SEED)
train_size = int(0.9 * len(full_train_set))
val_size   = len(full_train_set) - train_size
train_set, val_set = torch.utils.data.random_split(
    full_train_set, [train_size, val_size], generator=generator
)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"MNIST — train: {train_size:,}  val: {val_size:,}  test: {len(test_set):,}")

# ── EDA: sample grid ──────────────────────────────────────────────────────────
fig_eda, axes_eda = plt.subplots(2, 8, figsize=(14, 4))
for ax_e, (img_e, lbl_e) in zip(axes_eda.ravel(),
                                  [full_train_set[i] for i in range(16)]):
    ax_e.imshow(img_e.squeeze().numpy(), cmap="gray")
    ax_e.set_title(DIGIT_NAMES[lbl_e], fontsize=10)
    ax_e.axis("off")
fig_eda.suptitle("MNIST Sample Images", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── EDA: class distribution ───────────────────────────────────────────────────
all_labels   = [full_train_set[i][1] for i in range(len(full_train_set))]
label_counts = np.bincount(all_labels)
fig_bar, ax_bar = plt.subplots(figsize=(9, 3))
ax_bar.bar(range(NUM_CLASSES), label_counts, color=plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES)))
ax_bar.set_xticks(range(NUM_CLASSES))
ax_bar.set_xticklabels(DIGIT_NAMES)
ax_bar.set_ylabel("Count")
ax_bar.set_title("MNIST Class Distribution (Training Set)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Classes: {dict(zip(DIGIT_NAMES, label_counts.tolist()))}")

# ── Synthetic regression data with outliers ───────────────────────────────────
rng_reg   = np.random.default_rng(SEED)
n_reg     = 300
X_reg     = rng_reg.uniform(-3.0, 3.0, n_reg).astype(np.float32)
y_reg     = 2.0 * X_reg + 1.0 + rng_reg.normal(0.0, 0.5, n_reg).astype(np.float32)
# Add 15 outliers
outlier_mask = rng_reg.choice(n_reg, 15, replace=False)
y_reg[outlier_mask] += rng_reg.choice([-1, 1], 15) * rng_reg.uniform(5.0, 10.0, 15)

fig_reg, ax_reg = plt.subplots(figsize=(8, 4))
ax_reg.scatter(X_reg, y_reg, s=12, alpha=0.5, c="steelblue", label="Normal")
ax_reg.scatter(X_reg[outlier_mask], y_reg[outlier_mask], s=40,
               c="red", zorder=5, label="Outliers")
ax_reg.set_xlabel("x"); ax_reg.set_ylabel("y")
ax_reg.set_title("Synthetic Regression Data with Outliers (red)", fontsize=11, fontweight="bold")
ax_reg.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"Regression: n={n_reg}  outliers={len(outlier_mask)}")

---
## Part 1 — Loss Functions from Scratch

### 1.1 Loss Function Overview

| Loss | Formula | Task | Robust to outliers? |
|------|---------|------|---------------------|
| MSE | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Regression | No |
| MAE / L1 | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Regression | Yes |
| Huber | MSE if $|r| \leq \delta$, else $\delta(|r| - \tfrac{1}{2}\delta)$ | Regression | Partial |
| Cross-Entropy | $-\frac{1}{n}\sum \log p(y_i)$ | Classification | — |
| Focal | $-\frac{1}{n}\sum (1-p_t)^\gamma \log p_t$ | Imbalanced class. | — |

The key insight is that each loss imposes a **different gradient signal**: MSE penalises large errors quadratically; MAE penalises all errors equally; Huber interpolates; cross-entropy is derived from maximum likelihood; focal loss re-weights cross-entropy to focus on hard examples.

### 1.2 Regression Losses: MSE, MAE, Huber

For a residual $r = y - \hat{y}$:

$$\text{MSE}: \mathcal{L} = r^2, \quad \nabla = 2r$$
$$\text{MAE}: \mathcal{L} = |r|, \quad \nabla = \text{sign}(r)$$
$$\text{Huber}: \mathcal{L} = \begin{cases} \frac{1}{2}r^2 & |r| \leq \delta \\ \delta(|r| - \frac{1}{2}\delta) & |r| > \delta \end{cases}$$

Huber is **quadratic near zero** (smooth, stable gradients) and **linear for large residuals** (bounded gradient, robust to outliers).

In [ ]:
# ── MSE (Mean Squared Error) ──────────────────────────────────────────────────

def mse_loss_np(
    predictions: np.ndarray,
    targets:     np.ndarray,
) -> float:
    '''Mean squared error: mean((y_hat - y)^2).

    Args:
        predictions: Predicted values, shape (n,) or (n, d).
        targets:     Ground-truth values, same shape as predictions.

    Returns:
        Scalar mean squared error.
    '''
    return float(np.mean((predictions - targets) ** 2))


def mse_grad_np(
    predictions: np.ndarray,
    targets:     np.ndarray,
) -> np.ndarray:
    '''Gradient of MSE w.r.t. predictions: 2 * (y_hat - y) / n.

    Args:
        predictions: Predicted values.
        targets:     Ground-truth values.

    Returns:
        Gradient array, same shape as predictions.
    '''
    n = predictions.shape[0]
    return 2.0 * (predictions - targets) / n


# ── MAE (Mean Absolute Error / L1) ────────────────────────────────────────────

def mae_loss_np(
    predictions: np.ndarray,
    targets:     np.ndarray,
) -> float:
    '''Mean absolute error: mean(|y_hat - y|).

    Args:
        predictions: Predicted values.
        targets:     Ground-truth values.

    Returns:
        Scalar mean absolute error.
    '''
    return float(np.mean(np.abs(predictions - targets)))


def mae_grad_np(
    predictions: np.ndarray,
    targets:     np.ndarray,
) -> np.ndarray:
    '''Subgradient of MAE w.r.t. predictions: sign(y_hat - y) / n.

    Args:
        predictions: Predicted values.
        targets:     Ground-truth values.

    Returns:
        Gradient array with values in {-1/n, 0, +1/n}.
    '''
    n = predictions.shape[0]
    return np.sign(predictions - targets) / n


# ── Huber Loss ────────────────────────────────────────────────────────────────

def huber_loss_np(
    predictions: np.ndarray,
    targets:     np.ndarray,
    delta:       float = HUBER_DELTA,
) -> float:
    '''Huber loss: quadratic near zero, linear for large residuals.

    Args:
        predictions: Predicted values.
        targets:     Ground-truth values.
        delta:       Threshold between quadratic and linear regimes.

    Returns:
        Scalar Huber loss value.
    '''
    residuals  = predictions - targets
    abs_resid  = np.abs(residuals)
    quad_mask  = abs_resid <= delta
    loss_quad  = 0.5 * residuals[quad_mask] ** 2
    loss_lin   = delta * (abs_resid[~quad_mask] - 0.5 * delta)
    all_losses = np.empty_like(residuals)
    all_losses[quad_mask]  = loss_quad
    all_losses[~quad_mask] = loss_lin
    return float(all_losses.mean())


def huber_grad_np(
    predictions: np.ndarray,
    targets:     np.ndarray,
    delta:       float = HUBER_DELTA,
) -> np.ndarray:
    '''Gradient of Huber loss w.r.t. predictions.

    Args:
        predictions: Predicted values.
        targets:     Ground-truth values.
        delta:       Huber transition threshold.

    Returns:
        Gradient array: residual in quadratic zone, delta*sign elsewhere.
    '''
    n         = predictions.shape[0]
    residuals = predictions - targets
    abs_resid = np.abs(residuals)
    grad      = np.where(abs_resid <= delta, residuals, delta * np.sign(residuals))
    return grad / n


# ── Visualise: loss and gradient vs residual ──────────────────────────────────
residuals_plot = np.linspace(-4.0, 4.0, 400)
zero_targets   = np.zeros(400)

reg_loss_fns = [
    ("MSE",   lambda r: mse_loss_np(r, zero_targets),   lambda r: mse_grad_np(r, zero_targets)   * 400, "#1f77b4"),
    ("MAE",   lambda r: mae_loss_np(r, zero_targets),   lambda r: mae_grad_np(r, zero_targets)   * 400, "#ff7f0e"),
    ("Huber", lambda r: huber_loss_np(r, zero_targets), lambda r: huber_grad_np(r, zero_targets) * 400, "#2ca02c"),
]

fig_rl, (ax_loss, ax_grad) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, loss_fn, grad_fn, color in reg_loss_fns:
    loss_vals = np.array([loss_fn(np.array([r])) for r in residuals_plot])
    grad_vals = np.array([grad_fn(np.array([r])) for r in residuals_plot])
    ax_loss.plot(residuals_plot, loss_vals, color=color, lw=2, label=name)
    ax_grad.plot(residuals_plot, grad_vals, color=color, lw=2, label=name)

ax_loss.axhline(0, color="k", lw=0.6, ls="--", alpha=0.4)
ax_loss.axvline(0, color="k", lw=0.6, ls="--", alpha=0.4)
ax_loss.axvline(-HUBER_DELTA, color="#2ca02c", lw=0.8, ls=":", alpha=0.6)
ax_loss.axvline( HUBER_DELTA, color="#2ca02c", lw=0.8, ls=":", alpha=0.6)
ax_loss.set_xlabel("Residual r = ŷ - y"); ax_loss.set_ylabel("Loss value")
ax_loss.set_title("Regression Loss vs Residual", fontsize=11, fontweight="bold")
ax_loss.legend(fontsize=9)

ax_grad.axhline(0, color="k", lw=0.6, ls="--", alpha=0.4)
ax_grad.axvline(0, color="k", lw=0.6, ls="--", alpha=0.4)
ax_grad.set_xlabel("Residual r = ŷ - y"); ax_grad.set_ylabel("Gradient magnitude")
ax_grad.set_title("Loss Gradient vs Residual", fontsize=11, fontweight="bold")
ax_grad.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Robustness summary ────────────────────────────────────────────────────────
print("Gradient at residual = 5 (large error):")
r5 = np.array([5.0])
print(f"  MSE   gradient: {mse_grad_np(r5, np.zeros(1))[0]:.4f}")
print(f"  MAE   gradient: {mae_grad_np(r5, np.zeros(1))[0]:.4f}")
print(f"  Huber gradient: {huber_grad_np(r5, np.zeros(1))[0]:.4f}")
print(f"MSE gradient grows linearly — outliers dominate. MAE and Huber cap the gradient.")

### 1.3 Classification Losses: Cross-Entropy and Focal

**Multiclass cross-entropy** is derived from maximum likelihood under a categorical distribution:
$$\mathcal{L}_{\text{CE}} = -\frac{1}{n}\sum_{i=1}^n \log p(y_i | \mathbf{x}_i) = -\frac{1}{n}\sum_i \log \hat{p}_{i, y_i}$$

where $\hat{p} = \text{softmax}(\mathbf{z})$.

**Focal loss** (Lin et al., 2017) addresses class imbalance by adding a modulating factor $(1 - p_t)^\gamma$ that down-weights easy examples (high confidence):
$$\mathcal{L}_{\text{focal}} = -\frac{1}{n}\sum_i (1 - p_{t_i})^\gamma \log p_{t_i}$$

When $\gamma = 0$, focal loss reduces to cross-entropy.

In [ ]:
# ── Cross-Entropy from scratch ────────────────────────────────────────────────

def softmax_np(logits: np.ndarray) -> np.ndarray:
    '''Numerically stable softmax over the last axis.

    Args:
        logits: Array of shape (n, C) containing unnormalised log-probabilities.

    Returns:
        Probability array of shape (n, C) with rows summing to 1.
    '''
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_z   = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def cross_entropy_np(
    logits: np.ndarray,
    labels: np.ndarray,
) -> float:
    '''Multiclass cross-entropy loss.

    Applies softmax internally; expects raw logits, not probabilities.

    Args:
        logits: Array of shape (n, C) — unnormalised class scores.
        labels: Integer array of shape (n,) — class indices in [0, C).

    Returns:
        Scalar cross-entropy loss.
    '''
    n     = logits.shape[0]
    probs = softmax_np(logits)
    log_p = np.log(probs[np.arange(n), labels] + 1e-12)
    return float(-log_p.mean())


def cross_entropy_grad_np(
    logits: np.ndarray,
    labels: np.ndarray,
) -> np.ndarray:
    '''Gradient of cross-entropy loss w.r.t. logits.

    The combined softmax + CE gradient is: (p - one_hot(y)) / n.

    Args:
        logits: Array of shape (n, C).
        labels: Integer array of shape (n,).

    Returns:
        Gradient array of shape (n, C).
    '''
    n     = logits.shape[0]
    probs = softmax_np(logits).copy()
    probs[np.arange(n), labels] -= 1.0
    return probs / n


# ── Focal Loss from scratch ────────────────────────────────────────────────────

def focal_loss_np(
    logits: np.ndarray,
    labels: np.ndarray,
    gamma:  float = FOCAL_GAMMA,
) -> float:
    '''Multiclass focal loss.

    Focal(pt) = -(1 - pt)^gamma * log(pt), where pt is the model probability
    for the correct class.  gamma=0 reduces to standard cross-entropy.

    Args:
        logits: Array of shape (n, C) — unnormalised class scores.
        labels: Integer array of shape (n,) — class indices.
        gamma:  Focusing parameter. Higher gamma down-weights easy examples more.

    Returns:
        Scalar focal loss value.
    '''
    n          = logits.shape[0]
    probs      = softmax_np(logits)
    pt         = probs[np.arange(n), labels]
    focal_w    = (1.0 - pt) ** gamma
    log_pt     = np.log(pt + 1e-12)
    return float(-(focal_w * log_pt).mean())


# ── Loss as a function of prediction confidence ───────────────────────────────
# For a single binary problem: p = predicted probability of the correct class
p_vals = np.linspace(0.01, 0.999, 500)
ce_vals = -np.log(p_vals)
gamma_vals_plot = [0.5, 1.0, 2.0, 5.0]

fig_cl, (ax_ce, ax_grad_ce) = plt.subplots(1, 2, figsize=(13, 4.5))

ax_ce.plot(p_vals, ce_vals, color="#1f77b4", lw=2.5, label="CE (γ=0)")
for gamma_p, ls in zip(gamma_vals_plot, ["--", "-.", ":", "-"]):
    focal_p = -((1 - p_vals) ** gamma_p) * np.log(p_vals)
    ax_ce.plot(p_vals, focal_p, lw=2, ls=ls, label=f"Focal (γ={gamma_p})")

ax_ce.set_xlabel("p_correct (predicted probability of correct class)", fontsize=10)
ax_ce.set_ylabel("Loss value", fontsize=10)
ax_ce.set_title("Cross-Entropy vs Focal Loss", fontsize=11, fontweight="bold")
ax_ce.set_ylim(0, 5)
ax_ce.legend(fontsize=9)
ax_ce.axvline(0.5, color="k", lw=0.7, ls="--", alpha=0.4)

# Gradient of CE w.r.t. p_correct: d(-log p)/dp = -1/p
ce_grad   = -1.0 / p_vals
# Gradient of focal loss w.r.t. p_correct for gamma=2
g2        = 2.0
focal_g2  = -((1 - p_vals) ** g2) / p_vals + g2 * (1 - p_vals) ** (g2 - 1) * np.log(p_vals)

ax_grad_ce.plot(p_vals, np.clip(ce_grad, -30, 0), color="#1f77b4", lw=2, label="CE gradient")
ax_grad_ce.plot(p_vals, np.clip(focal_g2, -30, 0), color="#d62728", lw=2, ls="--",
                label="Focal (γ=2) gradient")
ax_grad_ce.set_xlabel("p_correct", fontsize=10)
ax_grad_ce.set_ylabel("dL/dp_correct", fontsize=10)
ax_grad_ce.set_title("Gradient Magnitude vs Prediction Confidence",
                     fontsize=11, fontweight="bold")
ax_grad_ce.legend(fontsize=9)
ax_grad_ce.set_ylim(-30, 0)

plt.tight_layout()
plt.show()

print("Key: focal loss gradient is smaller for high-confidence predictions (p close to 1),")
print("     directing the optimiser's attention to hard (low-confidence) examples.")
print(f"CE gradient at p=0.9:    {-1/0.9:.4f}")
print(f"CE gradient at p=0.1:    {-1/0.1:.4f}")
print(f"Focal(gamma=2) at p=0.9: {-((1-0.9)**2)/0.9 + 2*(1-0.9)*np.log(0.9):.4f}")
print(f"Focal(gamma=2) at p=0.1: {-((1-0.1)**2)/0.1 + 2*(1-0.1)*np.log(0.1):.4f}")

### 1.4 Loss Landscape Visualisation

For a 3-class classification problem we visualise the cross-entropy loss as a 2-D heat map over the simplex of predicted probabilities $(p_0, p_1, 1 - p_0 - p_1)$ for the true class 0. This illustrates the **non-linear response** of CE to probability mass allocation.

In [ ]:
# ── 2-D loss landscape over 3-class probability simplex ───────────────────────
resolution = 200
p0_grid    = np.linspace(0.01, 0.98, resolution)
p1_grid    = np.linspace(0.01, 0.98, resolution)
P0, P1     = np.meshgrid(p0_grid, p1_grid)
P2         = np.clip(1.0 - P0 - P1, 1e-6, 1.0)   # must be >=0

# Valid region: p0 + p1 <= 1
valid_mask  = (P0 + P1) <= 0.99
CE_landscape  = np.full((resolution, resolution), np.nan)
FOC_landscape = np.full((resolution, resolution), np.nan)

for i in range(resolution):
    for j in range(resolution):
        if valid_mask[i, j]:
            p = np.array([P0[i, j], P1[i, j], P2[i, j]])
            p = p / p.sum()  # renormalise to handle numerical edge cases
            CE_landscape[i, j]  = -np.log(p[0] + 1e-12)
            FOC_landscape[i, j] = -((1 - p[0]) ** FOCAL_GAMMA) * np.log(p[0] + 1e-12)

fig_ls, axes_ls = plt.subplots(1, 2, figsize=(13, 5))
for ax_ls, Z, title in zip(axes_ls,
                             [CE_landscape, FOC_landscape],
                             ["Cross-Entropy Loss (true class=0)",
                              f"Focal Loss γ={FOCAL_GAMMA} (true class=0)"]):
    im = ax_ls.contourf(P0, P1, np.clip(Z, 0, 6), levels=40, cmap="viridis")
    ax_ls.contour(P0, P1, Z, levels=[0.5, 1.0, 2.0, 4.0], colors="white", linewidths=0.7)
    plt.colorbar(im, ax=ax_ls, label="Loss value")
    ax_ls.set_xlabel("p₀ (prob. assigned to correct class 0)", fontsize=10)
    ax_ls.set_ylabel("p₁ (prob. assigned to incorrect class 1)", fontsize=10)
    ax_ls.set_title(title, fontsize=10, fontweight="bold")

fig_ls.suptitle("3-Class Loss Landscape over Probability Simplex",
                fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("Loss is lowest (dark blue) when p0 → 1 (all mass on correct class).")
print("Focal loss has a flatter landscape near high confidence — less gradient when already correct.")

# ── CE vs MSE on probability outputs ─────────────────────────────────────────
# When using MSE for classification (MSE on softmax(logits) vs one-hot),
# compare the gradient signal to CE
p_correct_range = np.linspace(0.01, 0.99, 500)
# CE: -log(p)
ce_loss_curve  = -np.log(p_correct_range)
# MSE on probability: (p - 1)^2 (for the correct class, 1-p for wrong)
# Simplified for single-class case: MSE ≈ (1 - p_correct)^2
mse_class_curve = (1.0 - p_correct_range) ** 2

fig_cm, ax_cm = plt.subplots(figsize=(8, 4))
ax_cm.plot(p_correct_range, ce_loss_curve,  color="#1f77b4", lw=2, label="Cross-Entropy")
ax_cm.plot(p_correct_range, mse_class_curve, color="#ff7f0e", lw=2, ls="--",
           label="MSE on softmax output")
ax_cm.set_xlabel("Predicted probability of correct class", fontsize=11)
ax_cm.set_ylabel("Loss value", fontsize=11)
ax_cm.set_title("CE vs MSE for Classification — Gradient Signal Comparison",
                fontsize=11, fontweight="bold")
ax_cm.legend(fontsize=9)
ax_cm.set_ylim(0, 5)
plt.tight_layout()
plt.show()
print("CE has a steeper gradient near p=0 (high loss region), giving stronger signal")
print("to severely misclassified examples — this is why CE dominates for classification.")

---
## Part 2 — PyTorch Loss Module Implementations

We wrap the from-scratch implementations as `nn.Module` classes so they integrate seamlessly with the standard training loop. PyTorch already provides `nn.MSELoss`, `nn.L1Loss`, `nn.HuberLoss`, and `nn.CrossEntropyLoss`; we only need to implement `FocalLoss`.

In [ ]:
# ── FocalLoss nn.Module ────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    '''Focal loss for multi-class classification.

    Focal loss re-weights cross-entropy to focus learning on hard, misclassified
    examples by down-weighting the loss for well-classified examples with a
    modulating factor (1 - p_t)^gamma.

    Reference: Lin et al. (2017) "Focal Loss for Dense Object Detection."

    Attributes:
        gamma:     Focusing parameter. gamma=0 reduces to cross-entropy.
        reduction: How to reduce across the batch ("mean" or "sum").
    '''

    def __init__(
        self,
        gamma:     float = FOCAL_GAMMA,
        reduction: str   = "mean",
    ) -> None:
        '''Initialise FocalLoss.

        Args:
            gamma:     Focusing parameter >= 0.
            reduction: "mean" (default) or "sum" over the batch.
        '''
        super().__init__()
        self.gamma     = gamma
        self.reduction = reduction

    def forward(
        self,
        logits:  torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        '''Compute focal loss from raw logits.

        Args:
            logits:  Tensor of shape (N, C) — unnormalised class scores.
            targets: Tensor of shape (N,)   — integer class indices in [0, C).

        Returns:
            Scalar focal loss (or per-sample tensor if reduction="none").
        '''
        log_probs = F.log_softmax(logits, dim=1)
        probs     = torch.exp(log_probs)
        log_pt    = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt        = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_w   = (1.0 - pt) ** self.gamma
        loss      = -focal_w * log_pt
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

    def __repr__(self) -> str:
        '''Return string representation.'''
        return f"FocalLoss(gamma={self.gamma}, reduction='{self.reduction}')"


# ── MSEClassificationLoss: MSE on softmax outputs vs one-hot targets ──────────

class MSEClassificationLoss(nn.Module):
    '''MSE loss applied to softmax probability outputs vs one-hot targets.

    This non-standard approach is used for comparison purposes only.
    Cross-entropy is strictly better for classification tasks.

    Attributes:
        num_classes: Number of output classes.
    '''

    def __init__(self, num_classes: int) -> None:
        '''Initialise MSEClassificationLoss.

        Args:
            num_classes: C, the total number of classes.
        '''
        super().__init__()
        self.num_classes = num_classes

    def forward(
        self,
        logits:  torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        '''Compute MSE between softmax(logits) and one-hot(targets).

        Args:
            logits:  Tensor of shape (N, C).
            targets: Integer class indices of shape (N,).

        Returns:
            Scalar MSE loss.
        '''
        probs    = F.softmax(logits, dim=1)
        one_hot  = F.one_hot(targets, self.num_classes).float()
        return F.mse_loss(probs, one_hot)


# ── Instantiate loss catalogue ────────────────────────────────────────────────
loss_catalogue: dict[str, nn.Module] = {
    "CrossEntropy": nn.CrossEntropyLoss(),
    "Focal":        FocalLoss(gamma=FOCAL_GAMMA),
    "MSE(class)":   MSEClassificationLoss(NUM_CLASSES),
}

for name, criterion in loss_catalogue.items():
    print(f"  {name}: {criterion}")

### 2.1 Numerical Sanity Check

We verify that our NumPy implementations match the PyTorch equivalents on random inputs.

In [ ]:
# ── NumPy vs PyTorch loss comparison ─────────────────────────────────────────
torch.manual_seed(SEED)
n_test    = 64
y_hat_t   = torch.randn(n_test, NUM_CLASSES)
y_labels  = torch.randint(0, NUM_CLASSES, (n_test,))
y_hat_reg = torch.randn(n_test)
y_tgt_reg = torch.randn(n_test)

# NumPy equivalents
y_hat_np      = y_hat_t.numpy()
y_labels_np   = y_labels.numpy()
y_hat_reg_np  = y_hat_reg.numpy()
y_tgt_reg_np  = y_tgt_reg.numpy()

print("NumPy vs PyTorch loss comparison:")
print("-" * 55)

# MSE
np_mse  = mse_loss_np(y_hat_reg_np, y_tgt_reg_np)
pt_mse  = nn.MSELoss()(y_hat_reg, y_tgt_reg).item()
diff_mse = abs(np_mse - pt_mse)
print(f"  MSE          — NumPy: {np_mse:.6f}  PyTorch: {pt_mse:.6f}  diff: {diff_mse:.2e}")

# MAE
np_mae  = mae_loss_np(y_hat_reg_np, y_tgt_reg_np)
pt_mae  = nn.L1Loss()(y_hat_reg, y_tgt_reg).item()
diff_mae = abs(np_mae - pt_mae)
print(f"  MAE          — NumPy: {np_mae:.6f}  PyTorch: {pt_mae:.6f}  diff: {diff_mae:.2e}")

# Huber
np_hub  = huber_loss_np(y_hat_reg_np, y_tgt_reg_np, delta=HUBER_DELTA)
pt_hub  = nn.HuberLoss(delta=HUBER_DELTA)(y_hat_reg, y_tgt_reg).item()
diff_hub = abs(np_hub - pt_hub)
print(f"  Huber        — NumPy: {np_hub:.6f}  PyTorch: {pt_hub:.6f}  diff: {diff_hub:.2e}")

# Cross-Entropy
np_ce   = cross_entropy_np(y_hat_np, y_labels_np)
pt_ce   = nn.CrossEntropyLoss()(y_hat_t, y_labels).item()
diff_ce = abs(np_ce - pt_ce)
print(f"  CrossEntropy — NumPy: {np_ce:.6f}  PyTorch: {pt_ce:.6f}  diff: {diff_ce:.2e}")

# Focal (compare our NumPy vs our PyTorch nn.Module)
np_foc  = focal_loss_np(y_hat_np, y_labels_np, gamma=FOCAL_GAMMA)
pt_foc  = FocalLoss(gamma=FOCAL_GAMMA)(y_hat_t, y_labels).item()
diff_foc = abs(np_foc - pt_foc)
print(f"  Focal        — NumPy: {np_foc:.6f}  PyTorch: {pt_foc:.6f}  diff: {diff_foc:.2e}")

print()
all_close = all(d < 1e-5 for d in [diff_mse, diff_mae, diff_hub, diff_ce, diff_foc])
print("All losses match within 1e-5. ✓" if all_close else "WARNING: mismatch detected.")

---
## Part 3 — Training Experiments

### 3.1 Classification on MNIST: CE vs Focal vs MSE

We train an identical 2-hidden-layer MLP with three different loss functions and compare training dynamics and final accuracy. Using MSE for classification is non-standard and serves as a baseline to show why cross-entropy is preferred.

In [ ]:
# ── Shared MLP architecture ────────────────────────────────────────────────────
class FlattenView(nn.Module):
    '''Flatten spatial dims inside nn.Sequential.'''

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Flatten all dims except batch.

        Args:
            x: Input tensor of shape (N, ...).

        Returns:
            Tensor of shape (N, prod(remaining dims)).
        '''
        return x.view(x.size(0), -1)


class MLP(nn.Module):
    '''Two-hidden-layer MLP for MNIST classification.

    Attributes:
        net: Sequential stack of layers.
    '''

    def __init__(
        self,
        input_dim:  int = INPUT_DIM,
        hidden_dim: int = HIDDEN_DIM,
        output_dim: int = NUM_CLASSES,
    ) -> None:
        '''Initialise MLP with ReLU activations.

        Args:
            input_dim:  Number of input features.
            hidden_dim: Width of each hidden layer.
            output_dim: Number of output logits.
        '''
        super().__init__()
        self.net = nn.Sequential(
            FlattenView(),
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Compute logits.

        Args:
            x: Input of shape (N, input_dim) or (N, C, H, W).

        Returns:
            Logits of shape (N, output_dim).
        '''
        return self.net(x)


def train_one_epoch(
    model:      nn.Module,
    dataloader: DataLoader,
    optimizer:  torch.optim.Optimizer,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Train for one epoch.

    Args:
        model:      Neural network.
        dataloader: Training DataLoader.
        optimizer:  Optimiser.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (avg_loss, accuracy).
    '''
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0
    for batch_inputs, batch_targets in dataloader:
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
        optimizer.zero_grad()
        outputs = model(batch_inputs)
        loss    = criterion(outputs, batch_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_inputs.size(0)
        _, predicted  = outputs.max(1)
        total        += batch_targets.size(0)
        correct      += predicted.eq(batch_targets).sum().item()
    return running_loss / total, correct / total


def evaluate(
    model:      nn.Module,
    dataloader: DataLoader,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Evaluate model.

    Args:
        model:      Neural network.
        dataloader: Validation or test DataLoader.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (avg_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct = 0
    total   = 0
    with torch.no_grad():
        for batch_inputs, batch_targets in dataloader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
            outputs      = model(batch_inputs)
            loss         = criterion(outputs, batch_targets)
            running_loss += loss.item() * batch_inputs.size(0)
            _, predicted  = outputs.max(1)
            total        += batch_targets.size(0)
            correct      += predicted.eq(batch_targets).sum().item()
    return running_loss / total, correct / total

In [ ]:
# ── Train MNIST with each classification loss ──────────────────────────────────
clf_results:   list[dict]      = []
clf_histories: dict[str, dict] = {}

for loss_name, criterion in loss_catalogue.items():
    model_clf = MLP().to(device)
    opt_clf   = torch.optim.Adam(model_clf.parameters(), lr=LEARNING_RATE)
    history: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }
    best_val_loss = float("inf")
    best_state    = None
    t_start       = time.time()

    for epoch in range(NUM_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(
            model_clf, train_loader, opt_clf, criterion, device
        )
        v_loss, v_acc = evaluate(model_clf, val_loader, criterion, device)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(v_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(v_acc)
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state    = {k: v.clone() for k, v in model_clf.state_dict().items()}
        if (epoch + 1) % 4 == 0 or epoch == 0:
            elapsed = time.time() - t_start
            print(f"[{loss_name:<14s}] Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"Train Loss: {tr_loss:.4f} | Val Loss: {v_loss:.4f} | "
                  f"Val Acc: {v_acc:.2%} | Time: {elapsed:.1f}s")

    model_clf.load_state_dict(best_state)
    _, test_acc = evaluate(model_clf, test_loader, criterion, device)
    elapsed_total = time.time() - t_start
    clf_results.append({
        "Loss":          loss_name,
        "Val Acc":       round(history["val_acc"][-1], 4),
        "Test Acc":      round(test_acc, 4),
        "Train Time(s)": round(elapsed_total, 1),
    })
    clf_histories[loss_name] = history
    print()

### 3.2 Regression with Outliers: MSE vs MAE vs Huber

We fit a simple linear model to the synthetic 1-D data (which contains 15 outliers) using gradient descent with each regression loss. This directly demonstrates the **robustness** difference: MSE gets pulled strongly toward outliers, MAE ignores them, and Huber achieves a balanced fit.

In [ ]:
# ── Simple linear regression via gradient descent ─────────────────────────────
class LinearModel(nn.Module):
    '''Single linear layer: y = w*x + b.

    Attributes:
        linear: The linear transformation.
    '''

    def __init__(self) -> None:
        '''Initialise with unit weight and zero bias.'''
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Apply linear transformation.

        Args:
            x: Input of shape (N, 1).

        Returns:
            Output of shape (N, 1).
        '''
        return self.linear(x)


X_reg_t = torch.tensor(X_reg, dtype=torch.float32).unsqueeze(1).to(device)
y_reg_t = torch.tensor(y_reg, dtype=torch.float32).unsqueeze(1).to(device)

reg_loss_catalogue: dict[str, nn.Module] = {
    "MSE":   nn.MSELoss(),
    "MAE":   nn.L1Loss(),
    "Huber": nn.HuberLoss(delta=HUBER_DELTA),
}

reg_fitted: dict[str, tuple[float, float]] = {}  # name → (w, b)

fig_rf, ax_rf = plt.subplots(figsize=(10, 5))
ax_rf.scatter(X_reg, y_reg, s=12, alpha=0.4, c="steelblue", zorder=2, label="Data")
ax_rf.scatter(X_reg[outlier_mask], y_reg[outlier_mask], s=50,
              c="red", zorder=3, label="Outliers", edgecolors="k", lw=0.5)

colors_reg = {"MSE": "#1f77b4", "MAE": "#ff7f0e", "Huber": "#2ca02c"}

for loss_name, reg_crit in reg_loss_catalogue.items():
    torch.manual_seed(SEED)
    lin_model = LinearModel().to(device)
    opt_lin   = torch.optim.Adam(lin_model.parameters(), lr=5e-3)
    for _ in range(NUM_EPOCHS_REG):
        lin_model.train()
        opt_lin.zero_grad()
        preds = lin_model(X_reg_t)
        loss  = reg_crit(preds, y_reg_t)
        loss.backward()
        opt_lin.step()
    w_fit = lin_model.linear.weight.item()
    b_fit = lin_model.linear.bias.item()
    reg_fitted[loss_name] = (w_fit, b_fit)
    x_line = np.linspace(-3, 3, 200)
    y_line = w_fit * x_line + b_fit
    ax_rf.plot(x_line, y_line, color=colors_reg[loss_name], lw=2,
               label=f"{loss_name}: y={w_fit:.2f}x+{b_fit:.2f}")

ax_rf.set_xlabel("x"); ax_rf.set_ylabel("y")
ax_rf.set_title("Linear Regression with Outliers — MSE vs MAE vs Huber",
                fontsize=11, fontweight="bold")
ax_rf.legend(fontsize=9)
ax_rf.set_ylim(-15, 15)
plt.tight_layout()
plt.show()

print("Fitted parameters (true: w=2.0, b=1.0):")
print(f"{'Loss':<8s}  {'w (true 2.0)':>14s}  {'b (true 1.0)':>14s}  {'|w-2|':>8s}")
print("-" * 52)
for loss_name, (w_fit, b_fit) in reg_fitted.items():
    print(f"{loss_name:<8s}  {w_fit:>14.4f}  {b_fit:>14.4f}  {abs(w_fit-2.0):>8.4f}")
print("\nMSE is pulled toward outliers (w deviation is largest).")
print("MAE and Huber stay closer to the true slope.")

### 3.2b Regression Learning Curves

We track the training loss of each regression loss over all 200 epochs and compare convergence speed and final loss values. We also evaluate RMSE on the **clean** data points (excluding outliers) to measure fit quality on the true relationship.


In [ ]:
# ── Regression training loss curves: MSE vs MAE vs Huber ─────────────────────
# Track loss at each epoch for NUM_EPOCHS_REG steps to visualise convergence.

reg_loss_histories: dict[str, list[float]] = {
    loss_name: [] for loss_name in reg_loss_catalogue
}

log_interval = 20  # record every 20 epochs

for loss_name, reg_crit in reg_loss_catalogue.items():
    torch.manual_seed(SEED)
    lm_curve = LinearModel().to(device)
    opt_curve = torch.optim.Adam(lm_curve.parameters(), lr=5e-3)
    for ep in range(NUM_EPOCHS_REG):
        lm_curve.train()
        opt_curve.zero_grad()
        loss_val = reg_crit(lm_curve(X_reg_t), y_reg_t)
        loss_val.backward()
        opt_curve.step()
        if (ep + 1) % log_interval == 0 or ep == 0:
            reg_loss_histories[loss_name].append(float(loss_val.item()))

x_axis_reg = [1] + list(range(log_interval, NUM_EPOCHS_REG + 1, log_interval))

fig_rc, ax_rc = plt.subplots(figsize=(9, 4))
for loss_name, color in colors_reg.items():
    ax_rc.plot(x_axis_reg, reg_loss_histories[loss_name],
               color=color, lw=2, marker="o", markersize=4, label=loss_name)

ax_rc.set_xlabel("Training Epoch", fontsize=11)
ax_rc.set_ylabel("Loss value", fontsize=11)
ax_rc.set_title("Regression Loss Convergence (data with 15 outliers)",
                fontsize=11, fontweight="bold")
ax_rc.legend(fontsize=9)
ax_rc.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# ── Final loss values and slope quality ───────────────────────────────────────
print("Final loss values and slope quality after training:")
print(f"{'Loss':<8s}  {'Final Loss':>12s}  {'Fitted w':>10s}  {'|w-2.0|':>10s}")
print("-" * 46)
for loss_name in reg_loss_catalogue:
    w_fit, b_fit = reg_fitted[loss_name]
    final_loss   = reg_loss_histories[loss_name][-1]
    print(f"{loss_name:<8s}  {final_loss:>12.4f}  {w_fit:>10.4f}  {abs(w_fit-2.0):>10.4f}")

print("\nNote: MSE achieves a lower MSE on this dataset (it minimises MSE by definition),")
print("but the slope is further from the true value 2.0 because outliers shift it.")
print("MAE/Huber have higher MSE values but more accurate slope estimates.")

# ── Compare predicted lines at clean data points only ─────────────────────────
clean_mask = np.ones(len(X_reg), dtype=bool)
clean_mask[outlier_mask] = False
X_clean = X_reg[clean_mask]
y_clean = y_reg[clean_mask]

print("\nRMSE on CLEAN data points (excluding outliers):")
for loss_name, (w_fit, b_fit) in reg_fitted.items():
    y_pred_clean = w_fit * X_clean + b_fit
    rmse_clean   = float(np.sqrt(np.mean((y_pred_clean - y_clean) ** 2)))
    print(f"  {loss_name:<8s}: RMSE = {rmse_clean:.4f}")
print("Lower RMSE on clean data = better fit to the true underlying relationship.")


In [ ]:
# ── Classification training curves ───────────────────────────────────────────
colors_clf = {"CrossEntropy": "#1f77b4", "Focal": "#ff7f0e", "MSE(class)": "#2ca02c"}
fig_tc, axes_tc = plt.subplots(1, 2, figsize=(13, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

for loss_name in loss_catalogue:
    hist  = clf_histories[loss_name]
    color = colors_clf[loss_name]
    axes_tc[0].plot(epochs_range, hist["val_acc"],  color=color, lw=2,
                    marker="o", markersize=5, label=loss_name)
    axes_tc[1].plot(epochs_range, hist["train_loss"], color=color, lw=2,
                    marker="s", markersize=5, label=loss_name)

axes_tc[0].set_xlabel("Epoch"); axes_tc[0].set_ylabel("Validation Accuracy")
axes_tc[0].set_title("Val Accuracy by Loss Function", fontsize=11, fontweight="bold")
axes_tc[0].legend(fontsize=9)
axes_tc[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

axes_tc[1].set_xlabel("Epoch"); axes_tc[1].set_ylabel("Training Loss (raw value)")
axes_tc[1].set_title("Training Loss by Loss Function", fontsize=11, fontweight="bold")
axes_tc[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

clf_df = pd.DataFrame(clf_results).sort_values("Val Acc", ascending=False)
print("Classification results (sorted by Val Acc):")
print(clf_df.to_string(index=False))

### 3.3 Imbalanced Classification: When Focal Loss Helps

We create an artificially imbalanced MNIST dataset where digit-0 makes up only 1% of training examples, then compare cross-entropy vs focal loss (gamma=2 and gamma=5) on recall for the minority class (digit 0). Higher gamma down-weights the majority classes more aggressively, forcing the model to attend to the rare class.


In [ ]:
# ── Simulate class imbalance on MNIST (digit 0 under-represented) ─────────────
# Keep all samples for digits 1-9 but only 1% of digit-0 samples
# to create a 99:1 imbalance. Then compare CE vs Focal.

rng_imb = np.random.default_rng(SEED + 7)

# Collect full training data as numpy arrays
all_imgs_list:   list[np.ndarray] = []
all_labels_list: list[int]        = []
for img_t, lbl in full_train_set:
    all_imgs_list.append(img_t.numpy())
    all_labels_list.append(int(lbl))

all_imgs_np   = np.stack(all_imgs_list, axis=0)    # (60000, 1, 28, 28)
all_labels_np = np.array(all_labels_list)           # (60000,)

# Select imbalanced subset: all non-zero digits + 1% of digit-0
zero_idx    = np.where(all_labels_np == 0)[0]
nonzero_idx = np.where(all_labels_np != 0)[0]
n_zero_keep = max(1, int(0.01 * len(zero_idx)))   # 1% of digit-0
kept_zero   = rng_imb.choice(zero_idx, n_zero_keep, replace=False)
imb_idx     = np.concatenate([nonzero_idx, kept_zero])
rng_imb.shuffle(imb_idx)

X_imb = torch.tensor(all_imgs_np[imb_idx],   dtype=torch.float32)
y_imb = torch.tensor(all_labels_np[imb_idx], dtype=torch.long)
print(f"Imbalanced dataset: {len(X_imb):,} samples")
print(f"  Digit 0: {int((y_imb == 0).sum()):,}   (1% of original {len(zero_idx):,})")
print(f"  Digits 1-9: {int((y_imb != 0).sum()):,}")

imb_ds     = torch.utils.data.TensorDataset(X_imb, y_imb)
imb_loader = DataLoader(imb_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=torch.cuda.is_available())

# ── Train CE and Focal on imbalanced data ─────────────────────────────────────
imb_losses: dict[str, nn.Module] = {
    "CrossEntropy": nn.CrossEntropyLoss(),
    "Focal(g=2)":   FocalLoss(gamma=2.0),
    "Focal(g=5)":   FocalLoss(gamma=5.0),
}

imb_results: list[dict] = []
num_epochs_imb = 5

for loss_name, crit_imb in imb_losses.items():
    torch.manual_seed(SEED)
    model_imb = MLP().to(device)
    opt_imb   = torch.optim.Adam(model_imb.parameters(), lr=LEARNING_RATE)
    for epoch in range(num_epochs_imb):
        model_imb.train()
        for xb, yb in imb_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt_imb.zero_grad()
            loss_imb = crit_imb(model_imb(xb), yb)
            loss_imb.backward()
            opt_imb.step()

    # Evaluate per-class recall on full test set
    model_imb.eval()
    correct_per_class = np.zeros(NUM_CLASSES, dtype=int)
    total_per_class   = np.zeros(NUM_CLASSES, dtype=int)
    with torch.no_grad():
        for imgs_te, lbls_te in test_loader:
            imgs_te, lbls_te = imgs_te.to(device), lbls_te.to(device)
            preds_te = model_imb(imgs_te).argmax(dim=1)
            for c in range(NUM_CLASSES):
                mask_c = lbls_te == c
                correct_per_class[c] += int((preds_te[mask_c] == c).sum())
                total_per_class[c]   += int(mask_c.sum())

    recall_digit0 = correct_per_class[0] / max(total_per_class[0], 1)
    overall_acc   = correct_per_class.sum() / total_per_class.sum()
    imb_results.append({
        "Loss":         loss_name,
        "Digit-0 Recall": round(recall_digit0, 4),
        "Overall Acc":  round(overall_acc, 4),
    })
    print(f"[{loss_name:<14s}] Digit-0 Recall: {recall_digit0:.2%}  "
          f"Overall: {overall_acc:.2%}")

imb_df = pd.DataFrame(imb_results).sort_values("Digit-0 Recall", ascending=False)
print("\nImbalanced MNIST — CE vs Focal Loss (digit-0 recall):")
print(imb_df.to_string(index=False))
print("\nFocal loss with higher gamma boosts recall on the minority class (digit 0)")
print("by reducing the gradient contribution from the easy majority classes.")


---
## Part 4 — Evaluation & Analysis

### 4.1 Outlier Sensitivity Analysis

We systematically vary the **outlier magnitude** and measure the bias it introduces in the fitted slope for each loss function. This quantifies robustness concretely.

In [ ]:
# ── Outlier sensitivity: slope bias vs outlier magnitude ─────────────────────
outlier_magnitudes = [0, 1, 2, 3, 5, 8, 12, 20]
true_slope         = 2.0

sensitivity_rows: list[dict] = []

for outlier_mag in outlier_magnitudes:
    rng_s   = np.random.default_rng(SEED + outlier_mag)
    n_s     = 200
    X_s     = rng_s.uniform(-3.0, 3.0, n_s).astype(np.float32)
    y_s     = 2.0 * X_s + 1.0 + rng_s.normal(0.0, 0.5, n_s).astype(np.float32)
    out_idx = rng_s.choice(n_s, 10, replace=False)
    y_s[out_idx] += outlier_mag * rng_s.choice([-1, 1], 10)
    X_s_t   = torch.tensor(X_s).unsqueeze(1).to(device)
    y_s_t   = torch.tensor(y_s).unsqueeze(1).to(device)
    row     = {"Outlier Magnitude": outlier_mag}
    for loss_name, reg_crit in reg_loss_catalogue.items():
        torch.manual_seed(SEED)
        lm   = LinearModel().to(device)
        opt  = torch.optim.Adam(lm.parameters(), lr=5e-3)
        for _ in range(NUM_EPOCHS_REG):
            lm.train(); opt.zero_grad()
            loss = reg_crit(lm(X_s_t), y_s_t)
            loss.backward(); opt.step()
        w_s      = lm.linear.weight.item()
        row[f"{loss_name} slope"] = round(w_s, 3)
        row[f"{loss_name} bias"]  = round(abs(w_s - true_slope), 4)
    sensitivity_rows.append(row)

sens_df = pd.DataFrame(sensitivity_rows)
print("Slope estimate vs outlier magnitude (true slope = 2.0):")
bias_cols = [c for c in sens_df.columns if "bias" in c]
print(sens_df[["Outlier Magnitude"] + [c for c in sens_df.columns if "slope" in c]].to_string(index=False))

# ── Plot slope bias ────────────────────────────────────────────────────────────
fig_sens, ax_sens = plt.subplots(figsize=(9, 4))
for loss_name, color in colors_reg.items():
    bias_vals = [row[f"{loss_name} bias"] for row in sensitivity_rows]
    ax_sens.plot(outlier_magnitudes, bias_vals,
                 color=color, lw=2, marker="o", markersize=6, label=loss_name)

ax_sens.set_xlabel("Outlier Magnitude", fontsize=11)
ax_sens.set_ylabel("|Fitted slope − True slope|", fontsize=11)
ax_sens.set_title("Slope Bias vs Outlier Magnitude", fontsize=11, fontweight="bold")
ax_sens.legend(fontsize=9)
ax_sens.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print("\nKey: MSE bias grows rapidly with outlier size (quadratic penalty magnifies them).")
print("     MAE bias stays nearly flat — constant gradient caps outlier influence.")
print("     Huber interpolates: behaves like MSE for small outliers, MAE for large ones.")

### 4.2 Gradient Magnitude Analysis

We compare the gradient magnitude each loss assigns to predictions at different confidence levels and error magnitudes, confirming the theoretical predictions.

In [ ]:
# ── Regression gradient magnitudes at different residual scales ───────────────
residual_vals = np.array([-5, -3, -2, -1, -0.5, 0, 0.5, 1, 2, 3, 5], dtype=np.float32)
print("Gradient magnitude per loss at various residual values (single sample, n=1):")
print(f"{'Residual':>10s}  {'MSE grad':>10s}  {'MAE grad':>10s}  {'Huber grad':>12s}")
print("-" * 48)
for r in residual_vals:
    g_mse   = mse_grad_np(np.array([r]), np.zeros(1))[0]
    g_mae   = mae_grad_np(np.array([r]), np.zeros(1))[0]
    g_huber = huber_grad_np(np.array([r]), np.zeros(1))[0]
    print(f"{r:>10.2f}  {g_mse:>10.4f}  {g_mae:>10.4f}  {g_huber:>12.4f}")

print(f"\n(Huber transition at delta={HUBER_DELTA})")

# ── CE gradient at different confidence levels ────────────────────────────────
print("\nCross-entropy gradient at predicted p_correct (n=1 binary case):")
print(f"{'p_correct':>10s}  {'CE loss':>10s}  {'|dL/dp|':>10s}")
print("-" * 35)
for p_c in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    loss_ce = -np.log(p_c)
    grad_ce = abs(-1.0 / p_c)
    print(f"{p_c:>10.2f}  {loss_ce:>10.4f}  {grad_ce:>10.4f}")

# ── Visualise gradient flow per loss ──────────────────────────────────────────
fig_ga, axes_ga = plt.subplots(1, 2, figsize=(13, 4.5))

r_range = np.linspace(-4, 4, 300)
axes_ga[0].plot(r_range, mse_grad_np(r_range, np.zeros(300)) * 300,
                color="#1f77b4", lw=2, label="MSE")
axes_ga[0].plot(r_range, mae_grad_np(r_range, np.zeros(300)) * 300,
                color="#ff7f0e", lw=2, label="MAE")
axes_ga[0].plot(r_range, huber_grad_np(r_range, np.zeros(300)) * 300,
                color="#2ca02c", lw=2, label="Huber")
axes_ga[0].axvline(-HUBER_DELTA, color="#2ca02c", lw=0.8, ls=":", alpha=0.6)
axes_ga[0].axvline( HUBER_DELTA, color="#2ca02c", lw=0.8, ls=":", alpha=0.6)
axes_ga[0].axhline(0, color="k", lw=0.6, ls="--", alpha=0.4)
axes_ga[0].set_xlabel("Residual r"); axes_ga[0].set_ylabel("Gradient")
axes_ga[0].set_title("Regression Loss Gradients", fontsize=11, fontweight="bold")
axes_ga[0].legend(fontsize=9)

p_range = np.linspace(0.01, 0.99, 300)
for gam, color, ls in [(0, "#1f77b4", "-"), (1, "#ff7f0e", "--"),
                        (2, "#2ca02c", "-."), (5, "#d62728", ":")]:
    focal_deriv = -((1 - p_range) ** gam) / p_range
    if gam > 0:
        focal_deriv += gam * (1 - p_range) ** (gam - 1) * np.log(p_range + 1e-12)
    axes_ga[1].plot(p_range, np.clip(focal_deriv, -40, 0), color=color, lw=2,
                    ls=ls, label=f"Focal γ={gam}" if gam > 0 else f"CE (γ={gam})")

axes_ga[1].set_xlabel("p_correct"); axes_ga[1].set_ylabel("dL/dp_correct")
axes_ga[1].set_title("CE/Focal Gradient vs Confidence", fontsize=11, fontweight="bold")
axes_ga[1].legend(fontsize=9); axes_ga[1].set_ylim(-40, 0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comprehensive comparison table ───────────────────────────────────────────
comparison_rows = [
    {
        "Loss":          "MSE",
        "Task":          "Regression",
        "Gradient":      "Linear (2r)",
        "Robust":        "No",
        "PyTorch":       "nn.MSELoss",
        "Notes":         "Sensitive to outliers; large errors dominate",
    },
    {
        "Loss":          "MAE",
        "Task":          "Regression",
        "Gradient":      "Constant (sign(r))",
        "Robust":        "Yes",
        "PyTorch":       "nn.L1Loss",
        "Notes":         "Non-differentiable at 0; robust but slow convergence",
    },
    {
        "Loss":          "Huber",
        "Task":          "Regression",
        "Gradient":      "Linear near 0, capped",
        "Robust":        "Partial",
        "PyTorch":       "nn.HuberLoss",
        "Notes":         f"Balances MSE+MAE; delta={HUBER_DELTA} controls threshold",
    },
    {
        "Loss":          "Cross-Entropy",
        "Task":          "Classification",
        "Gradient":      "p_hat - one_hot(y)",
        "Robust":        "N/A",
        "PyTorch":       "nn.CrossEntropyLoss",
        "Notes":         "Derived from max likelihood; standard for classification",
    },
    {
        "Loss":          "Focal",
        "Task":          "Imbalanced class.",
        "Gradient":      "Downweighted CE",
        "Robust":        "N/A",
        "PyTorch":       "Custom (FocalLoss)",
        "Notes":         f"gamma={FOCAL_GAMMA}; focuses on hard examples",
    },
]

comp_df = pd.DataFrame(comparison_rows)
print("Loss Function Comparison:")
print(comp_df.to_string(index=False))
print()
print("Classification results on MNIST:")
print(pd.DataFrame(clf_results).sort_values("Val Acc", ascending=False).to_string(index=False))

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Cross-entropy is the right choice for classification**: its gradient $(\hat{p} - y)$    gives strong signal to misclassified examples and is derived directly from maximum    likelihood; MSE on softmax outputs has a weaker, saturating gradient and converges slower.
2. **MSE is sensitive to outliers**: its quadratic penalty makes large errors dominate the    gradient update, pulling the model toward extreme values. MAE and Huber are more robust.
3. **Huber loss is the practitioner's choice for regression**: by being quadratic near zero    and linear far from zero, it gets the smoothness of MSE near the optimum while capping    the influence of large residuals.
4. **Focal loss shifts attention to hard examples**: by multiplying cross-entropy by    $(1-p_t)^\gamma$, well-classified examples contribute less to the gradient — this is    critical for heavily imbalanced datasets but has modest effect when classes are balanced.
5. **Gradient geometry shapes the learning trajectory**: the choice of loss determines    *which* errors the model is penalised for most — understanding the gradient as a function    of the prediction is the key to choosing the right loss for a given problem.

### What's Next

→ **05-05 (Regularisation)** adds Dropout, BatchNorm, and L1/L2 weight decay on top of the training loops established here, showing how to prevent overfitting.

### Going Further

- Lin et al. (2017) — *Focal Loss for Dense Object Detection* (RetinaNet).
- Huber, P. J. (1964) — *Robust Estimation of a Location Parameter* (original Huber paper).
- Bishop (2006), Pattern Recognition and Machine Learning — Chapter 5 covers loss functions   from a probabilistic perspective.